In [ ]:
import numpy as np
from scipy.special import gamma, gammaln
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# PARAMETER MODEL
# ============================================================
lam    = 1.111111    # intensitas klaim (lambda)
alpha  = 2.145432   # shape parameter distribusi Gamma
beta2  = 0.009844   # rate parameter distribusi Gamma
tetha  = 0
c      = (1+tetha)*242.17     # premi / kontribusi peserta
mu     = 217.95     # rata-rata besar klaim

# ============================================================
# LANGKAH 1: HITUNG KONSTANTA DASAR
# ============================================================
print("=" * 65)
print("LANGKAH 1: KONSTANTA DASAR")
print("=" * 65)

# Peluang survival awal
phi_0 = 1 - (lam * mu) / c
print(f"\nφ(0) = 1 - λμ/c")
print(f"     = 1 - ({lam} × {mu}) / {c}")
print(f"     = 1 - {lam * mu:.5f} / {c}")
print(f"     = {phi_0:.6f}")
print(f"  → Makna: peluang survive saat modal awal u=0 hanya {phi_0*100:.4f}%")

# Basis koefisien deret
beta2_alpha = beta2 ** alpha
base_coef   = lam * beta2_alpha / c
print(f"\nλβ₂ᵅ / c = {lam} × ({beta2})^{alpha} / {c}")
print(f"         = {lam} × {beta2_alpha:.8e} / {c}")
print(f"         = {base_coef:.6e}")
print(f"  → Sangat kecil karena β₂ << 1, deret konvergen cepat")

# Basis argumen Mittag-Leffler
ml_base = beta2 + lam / c
print(f"\nβ₂ + λ/c = {beta2} + {lam}/{c}")
print(f"         = {beta2} + {lam/c:.6f}")
print(f"         = {ml_base:.6f}")
print(f"  → Laju peluruhan gabungan skala klaim & rasio intensitas-premi")


# ============================================================
# FUNGSI BANTU: TURUNAN KE-k MITTAG-LEFFLER
# ============================================================
def mittag_leffler_deriv_k(k, z, n_terms=30):
    """
    Menghitung turunan ke-k dari fungsi Mittag-Leffler E^(k)_{1, alpha*k+1}(z)

    Rumus (dari paper, persamaan setelah (2)):
        E^(k)_{1, alpha*k+1}(z) = sum_{j=0}^{inf} [(j+k)! / (j! * Gamma(j + k*(alpha+1) + 1))] * z^j

    Parameter:
        k      : orde turunan (indeks deret luar)
        z      : argumen = (beta2 + lambda/c) * u
        n_terms: jumlah suku yang dihitung (default 30)
    """
    if k == 0:
        # E^(0)_{1,1}(z) = e^z
        return np.exp(z)

    result = 0.0
    for j in range(n_terms + 1):
        # log-numerator: ln(Gamma(j+k+1)) = ln((j+k)!)
        ln_num = gammaln(j + k + 1)
        # log-denominator: ln(j!) + ln(Gamma(j + k*(alpha+1) + 1))
        ln_den = gammaln(j + 1) + gammaln(j + k * (alpha + 1) + 1)

        if z <= 0:
            if j == 0:
                result += np.exp(ln_num - ln_den)
            continue

        # log-term lengkap
        ln_term = ln_num - ln_den + j * np.log(z)
        term = np.exp(ln_term)

        if not np.isfinite(term):
            break
        result += term

    return result


# ============================================================
# FUNGSI UTAMA: HITUNG psi(u)
# ============================================================
def hitung_psi_u(u, K=20, verbose=True):
    """
    Menghitung peluang kebangkrutan psi(u) dengan penjabaran tiap suku.

    psi(u) = 1 - phi(u)

    phi(u) = e^{-beta2*u} * phi(0) * sum_{k=0}^{K} [
                (-1)^k / k! * (lambda*beta2^alpha/c)^k
                * u^{(alpha+1)*k}
                * E^(k)_{1, alpha*k+1}[(beta2 + lambda/c)*u]
             ]

    Parameter:
        u      : modal awal perusahaan
        K      : jumlah suku deret (default 20)
        verbose: tampilkan penjabaran (default True)
    """
    if verbose:
        print("\n" + "=" * 65)
        print(f"LANGKAH 2 & 3: PERHITUNGAN SUKU DERET UNTUK u = {u}")
        print("=" * 65)

    # Faktor tetap di luar deret
    decay = np.exp(-beta2 * u)          # e^{-beta2 * u}
    z      = ml_base * u                 # argumen Mittag-Leffler

    if verbose:
        print(f"\nFaktor peluruhan e^(-β₂u) = e^(-{beta2}×{u})")
        print(f"                          = e^(-{beta2*u:.4f})")
        print(f"                          = {decay:.6e}")
        print(f"\nArgumen Mittag-Leffler z  = (β₂ + λ/c) × u")
        print(f"                          = {ml_base:.6f} × {u}")
        print(f"                          = {z:.4f}")
        print(f"\n{'k':>3} | {'(-1)^k/k!':>12} | {'(coef)^k':>12} | {'u^((α+1)k)':>14} | {'E^(k)(z)':>14} | {'nilai suku':>14} | tanda")
        print("-" * 90)

    S = 0.0  # akumulator jumlah deret

    for k in range(K + 1):
        # Komponen 1: (-1)^k / k!
        sign      = (-1) ** k
        fact_k    = np.exp(gammaln(k + 1))   # k!
        coef_sign = sign / fact_k

        # Komponen 2: (lambda * beta2^alpha / c)^k
        coef_pow = base_coef ** k

        # Komponen 3: u^{(alpha+1)*k}
        exp_u = (alpha + 1) * k
        u_pow = u ** exp_u if u > 0 else (1.0 if exp_u == 0 else 0.0)

        # Komponen 4: E^(k)_{1, alpha*k+1}(z)
        ml_val = mittag_leffler_deriv_k(k, z)

        # Nilai suku ke-k
        term = coef_sign * coef_pow * u_pow * ml_val

        if np.isfinite(term):
            S += term

        if verbose:
            tanda = "+" if sign > 0 else "-"
            print(f"{k:>3} | {coef_sign:>12.4e} | {coef_pow:>12.4e} | "
                  f"{u_pow:>14.4e} | {ml_val:>14.4e} | {abs(term):>14.4e} | {tanda}")

    # phi(u) dan psi(u)
    phi_u = phi_0 * decay * S
    psi_u = max(0.0, min(1.0, 1.0 - phi_u))

    return psi_u


# ============================================================
# JALANKAN UNTUK BEBERAPA NILAI u
# ============================================================
if __name__ == "__main__":

    # Contoh penjabaran lengkap untuk u = 1000
    psi = hitung_psi_u(u=1000, K=20, verbose=True)

LANGKAH 1: KONSTANTA DASAR

φ(0) = 1 - λμ/c
     = 1 - (1.111111 × 217.95) / 242.17
     = 1 - 242.16664 / 242.17
     = 0.000014
  → Makna: peluang survive saat modal awal u=0 hanya 0.0014%

λβ₂ᵅ / c = 1.111111 × (0.009844)^2.145432 / 242.17
         = 1.111111 × 4.94864321e-05 / 242.17
         = 2.270509e-07
  → Sangat kecil karena β₂ << 1, deret konvergen cepat

β₂ + λ/c = 0.009844 + 1.111111/242.17
         = 0.009844 + 0.004588
         = 0.014432
  → Laju peluruhan gabungan skala klaim & rasio intensitas-premi

LANGKAH 2 & 3: PERHITUNGAN SUKU DERET UNTUK u = 1000

Faktor peluruhan e^(-β₂u) = e^(-0.009844×1000)
                          = e^(-9.8440)
                          = 5.306463e-05

Argumen Mittag-Leffler z  = (β₂ + λ/c) × u
                          = 0.014432 × 1000
                          = 14.4321

  k |    (-1)^k/k! |     (coef)^k |     u^((α+1)k) |       E^(k)(z) |     nilai suku | tanda
----------------------------------------------------------------------------

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy.special import gammaln

# ============================================================
# PARAMETER MODEL
# ============================================================
lam    = 1.111111
alpha  = 2.145432
beta2  = 0.009844
mu     = 217.95
E_s    = 242.17  # Expected aggregate claims (new E(s) constant)

# ============================================================
# METODE A: DERET MITTAG-LEFFLER
# Theorem 3.1 Constantinescu et al.
# ============================================================
def mittag_leffler_k(k, z, n_terms=80):
    """
    Turunan ke-k dari E_{1, alpha*k+1}(z):
        E^(k)_{1,alpha*k+1}(z) = sum_{j=0}^inf [(j+k)!/(j! Gamma(j+k(a+1)+1))] z^j
    """
    if k == 0:
        return np.exp(min(z, 700))
    s = 0.0
    for j in range(n_terms + 1):
        ln_n = gammaln(j + k + 1)
        ln_d = gammaln(j + 1) + gammaln(j + k * (alpha + 1) + 1)
        lz   = j * np.log(z) if z > 1e-300 else (0.0 if j == 0 else -np.inf)
        lt   = ln_n - ln_d + lz
        if lt > 700 or not np.isfinite(lt):
            break
        s += np.exp(lt)
    return s

def phi_mittag_leffler(u, c, K=60):
    """
    phi(u) via deret Mittag-Leffler (Theorem 3.1):
        phi(u) = e^{-beta2*u} * phi(0) * sum_{k=0}^K [
                    (-1)^k/k! * (lam*beta2^alpha/c)^k
                    * u^{(alpha+1)k}
                    * E^(k)_{1,alpha*k+1}[(beta2+lam/c)*u] ]

    CATATAN: Berdasarkan uji rasio suku (lihat Bagian 4.5), deret ini
    konvergen absolut untuk SEMUA u >= 0, tidak dibatasi. Faktor yang
    bergantung pada k berperilaku ~ k^-(alpha+1) untuk k -> infinity,
    terlepas dari nilai u yang tetap. Karena itu tidak ada batas atas u
    yang diberlakukan di sini; K (banyak suku) yang menyesuaikan agar
    presisi tetap terjaga untuk u besar.
    """
    phi0    = 1.0 - E_s / c
    base_c  = lam * (beta2 ** alpha) / c
    ml_base = beta2 + lam / c
    decay   = np.exp(-beta2 * u)
    z       = ml_base * u

    log_pos, log_neg = [], []

    for k in range(K + 1):
        lf    = -gammaln(k + 1)                          # log(1/k!)
        lbc   = k * np.log(base_c) if base_c > 0 else -np.inf
        exp_u = (alpha + 1) * k
        lu    = exp_u * np.log(u) if u > 0 else (0.0 if exp_u == 0 else -np.inf)
        ml    = mittag_leffler_k(k, z)
        if ml <= 0 or not np.isfinite(ml):
            continue
        lml  = np.log(ml)
        lsk  = lf + lbc + lu + lml
        if not np.isfinite(lsk):
            continue
        (log_pos if k % 2 == 0 else log_neg).append(lsk)

    def lse(arr):
        if not arr:
            return -np.inf
        m = max(arr)
        return m + np.log(sum(np.exp(v - m) for v in arr))

    S_pos = np.exp(lse(log_pos)) if np.isfinite(lse(log_pos)) else 0.0
    S_neg = np.exp(lse(log_neg)) if np.isfinite(lse(log_neg)) else 0.0
    phi_u = phi0 * decay * (S_pos - S_neg)
    return float(np.clip(phi_u, 0.0, 1.0))

# ============================================================
# FUNGSI UTAMA
# ============================================================
def hitung_phi(u, c):
    return phi_mittag_leffler(u, c)

def hitung_psi(u, c):
    return 1.0 - hitung_phi(u, c)

# ============================================================
# SKENARIO
# ============================================================
theta_list = [0.05, 0.10, 0.20, 0.30, 0.50]
c_list     = [E_s * (1 + th) for th in theta_list]

# Tidak ada batas atas u — deret konvergen absolut untuk semua u >= 0
u_list = [0, 100, 500, 1000, 2500, 5000, 7500, 10000,
          15000, 20000, 30000, 40000, 50000]

# ============================================================
# CETAK PARAMETER
# ============================================================
print("=" * 70)
print("PARAMETER MODEL")
print("=" * 70)
print(f"  lambda  = {lam}")
print(f"  alpha   = {alpha}   (shape Gamma)")
print(f"  beta2   = {beta2}   (rate Gamma)")
print(f"  mu      = {mu}      (X_bar dari data)")
print(f"  E(s)    = {E_s:.4f}")
print("  Batas u : tidak ada (konvergen untuk semua u >= 0)")

# ============================================================
# TABEL SENSITIVITAS  (baris=u, kolom=theta)
# ============================================================
print(f"\n{'='*70}")
print("TABEL PELUANG KEBANGKRUTAN psi(u)")
print("Baris = Modal Awal u  |  Kolom = Safety Loading theta")
print(f"{'='*70}")

print(f"\n{'u':>8} |", end="")
for th in theta_list:
    print(f"   theta={th*100:.0f}%  |", end="")
print()
print("-" * 70)

hasil = {}
for u in u_list:
    print(f"{u:>8} |", end="")
    hasil[u] = {}
    for th, c in zip(theta_list, c_list):
        psi = hitung_psi(u, c)
        hasil[u][th] = psi
        print(f"   {psi*100:6.2f}%    |", end="")
    print()

print("\n" + "="*70)
print("CATATAN: Deret Mittag-Leffler konvergen absolut untuk seluruh u >= 0")
print("(lihat pembuktian rasio suku, Bagian 4.5).")
print("="*70)

PARAMETER MODEL
  lambda  = 1.111111
  alpha   = 2.145432   (shape Gamma)
  beta2   = 0.009844   (rate Gamma)
  mu      = 217.95      (X_bar dari data)
  E(s)    = 242.1700
  Batas u : tidak ada (konvergen untuk semua u >= 0)

TABEL PELUANG KEBANGKRUTAN psi(u)
Baris = Modal Awal u  |  Kolom = Safety Loading theta

       u |   theta=5%  |   theta=10%  |   theta=20%  |   theta=30%  |   theta=50%  |
----------------------------------------------------------------------
       0 |    95.24%    |    90.91%    |    83.33%    |    76.92%    |    66.67%    |
     100 |    92.85%    |    86.59%    |    76.21%    |    67.97%    |    55.73%    |
     500 |    82.48%    |    69.00%    |    50.08%    |    37.82%    |    23.62%    |
    1000 |    71.00%    |    51.75%    |    29.39%    |    17.95%    |     7.90%    |
    2500 |    45.30%    |    21.84%    |     5.95%    |     1.93%    |     0.30%    |
    5000 |     0.00%    |     0.00%    |     0.00%    |     0.00%    |     0.00%    |
    7500 |  